In [0]:
--==============================================================================================
-- Table Preview
--==============================================================================================
SELECT * FROM `workspace`.`default`.`tv_viewship` LIMIT 100;


--=============================================================================================
---Understanding The Data set
--=============================================================================================

-- FROM VIEWER TABLE

--- Date Range
SELECT 
    MIN(from_utc_timestamp(v.RecordDate2, 'Africa/Johannesburg')) AS Earliest,
    MAX(from_utc_timestamp(v.RecordDate2, 'Africa/Johannesburg')) AS Latest
FROM `workspace`.`default`.`tv_viewship` AS v
LEFT JOIN `workspace`.`default`.`TV_UserProfile` AS u
ON v.UserID0 = u.UserID;
-- 2016-01-01 TO 2016-04-01

-- Different Channels
SELECT DISTINCT Channel2 FROM `workspace`.`default`.`tv_viewship`;
-- 21 Different Channels.


-- FROM USER_PROFILE TABLE

--- Gender
SELECT DISTINCT Gender FROM `workspace`.`default`.`TV_UserProfile`;

--- Race
SELECT DISTINCT Race FROM `workspace`.`default`.`TV_UserProfile`;
-- We have white,black,coloured,other,None,indian_asian


---  Check the youngest Age and oldest
SELECT
MAX(Age) OLDEST,
MIN(Age) YOUNGEST
FROM `workspace`.`default`.`TV_UserProfile`;
--- Youngest 0 and oldest 114


--- Check Diiferent Types of Province Province
SELECT DISTINCT Province FROM `workspace`.`default`.`TV_UserProfile`;
--we have 9 south African Provinces and 1 Unknown



--==============================================================================================
-- Data Cleaning & Joining(Viewship Left joined to User_profile )
--==============================================================================================


-- 1. Converting Nones to Unknowns Good for PowerBI  
SELECT 
    v.UserID0,

    -- Cleaned Columns (Replace 'None' as Unkwown properly)
    CASE 
        WHEN u.Gender = 'None' OR u.Gender IS NULL THEN 'Unknown'
        ELSE u.Gender
    END AS Gender,

    CASE 
        WHEN u.Race = 'None' OR u.Race IS NULL THEN 'Unknown'
        ELSE u.Race
    END AS Race,

    CASE 
        WHEN u.Province = 'None' OR u.Province IS NULL THEN 'Unknown'
        ELSE u.Province
    END AS Province,

    CASE 
        WHEN v.Channel2 = 'None' OR v.Channel2 IS NULL THEN 'Unknown'
        ELSE v.Channel2
    END AS Channel,


    v.RecordDate2,
    v.`Duration 2`

FROM `workspace`.`default`.`tv_viewship` v
LEFT JOIN `workspace`.`default`.`TV_UserProfile` u 
ON v.UserID0 = u.UserID;


--- 2. CHECK FOR NULLS 
SELECT 
          v.UserID0,
          u.Race,
          u.Gender,
          u.Age,
          u.Province,
          v.Channel2 AS Channel

FROM          `workspace`.`default`.`tv_viewship` v
LEFT JOIN    `workspace`.`default`.`TV_UserProfile` u 
ON            v.UserID0 = u.UserID
WHERE Gender IS NULL OR 
      Race IS NULL OR 
      Age IS NULL OR 
      Province IS NULL OR 
      Channel2 IS NULL OR 
      `Duration 2` IS NULL OR
      RecordDate2 IS NULL OR
      v.UserID0 IS NULL;


 --- 3. Check For Duplicates
SELECT 
          v.UserID0,
          u.Race,
          u.Gender,
          u.Age,
          u.Province,
          v.Channel2 AS Channel

FROM          `workspace`.`default`.`tv_viewship` v
LEFT JOIN    `workspace`.`default`.`TV_UserProfile` u 
ON            v.UserID0 = u.UserID
WHERE (UserID0, RecordDate2) IN (
    SELECT UserID0, RecordDate2
    FROM `workspace`.`default`.`tv_viewship`
    GROUP BY UserID0, RecordDate2
    HAVING COUNT(*) > 1);
-- There are 22 duplicates


-- Calculate total and average watch minutes per user, then get max/min across users
WITH user_watch AS (
    SELECT
        v.UserID0,
        ROUND(
            AVG(
                (EXTRACT(HOUR FROM v.`Duration 2`) * 60) +
                EXTRACT(MINUTE FROM v.`Duration 2`)
            ), 2
        ) AS Avg_Watch_Minutes,
        SUM(
            (EXTRACT(HOUR FROM v.`Duration 2`) * 60) +
            EXTRACT(MINUTE FROM v.`Duration 2`)
        ) AS Total_Watch_Minutes
    FROM `workspace`.`default`.`tv_viewship` v
    LEFT JOIN `workspace`.`default`.`TV_UserProfile` u
        ON v.UserID0 = u.UserID
    GROUP BY v.UserID0
)
SELECT
    COUNT(*) AS View_Count,
    ROUND(AVG(Avg_Watch_Minutes), 2) AS Avg_Watch_Minutes,
    MAX(Total_Watch_Minutes) AS Max_Watch_Minutes,
    MIN(Total_Watch_Minutes) AS Min_Watch_Minutes
FROM user_watch;


--======================================================================================================================
-- FEATURE ENGINEERING & Exploratory Analysis (MODIFIED VERSION)
--======================================================================================================================

WITH profile AS (
    SELECT DISTINCT
        -- 1. IDENTIFIERS
        v.UserID0,
        
        -- 2. TIME DIMENSIONS (converted to SA timezone)
        from_utc_timestamp(v.RecordDate2, 'Africa/Johannesburg') AS RecordDate_SA,
        
        -- 3. DURATION & CHANNEL
        v.`Duration 2`,
        CASE 
            WHEN v.Channel2 IS NULL OR TRIM(v.Channel2) = '' OR v.Channel2 = 'None' THEN 'Unknown'
            ELSE v.Channel2
        END AS channel,
        
        -- 4. DEMOGRAPHICS
        CASE 
            WHEN u.Age BETWEEN 0 AND 114 THEN u.Age
            ELSE NULL
        END AS age,
        
        CASE 
            WHEN u.Gender IS NULL OR TRIM(u.Gender) = '' OR u.Gender = 'None' THEN 'Unknown'
            ELSE u.Gender
        END AS gender,

        CASE 
            WHEN u.Race IS NULL OR TRIM(u.Race) = '' OR u.Race = 'None' OR u.Race = 'other/none comined' THEN 'Unknown'
            ELSE u.Race
        END AS race,

        CASE 
            WHEN u.Province IS NULL OR TRIM(u.Province) = '' OR u.Province = 'None' THEN 'Unknown'
            ELSE u.Province
        END AS province

    FROM `workspace`.`default`.`tv_viewship` v
    LEFT JOIN `workspace`.`default`.`TV_UserProfile` u 
        ON v.UserID0 = u.UserID
    
    -- Remove duplicates by keeping only unique combinations
    WHERE (v.UserID0, v.RecordDate2, v.Channel2, v.`Duration 2`) IN (
        SELECT UserID0, RecordDate2, Channel2, `Duration 2`
        FROM `workspace`.`default`.`tv_viewship`
        GROUP BY UserID0, RecordDate2, Channel2, `Duration 2`
        HAVING COUNT(*) = 1
    )
)

SELECT 
--========================================================================================================
-- 1. IDENTIFIERS
--========================================================================================================
    UserID0,
    
--========================================================================================================
-- 2. DATE & TIME DIMENSIONS (from RecordDate_SA)
--========================================================================================================

    -- Full date components
    date_format(RecordDate_SA, 'yyyy-MM-dd') AS Full_date,
    date_format(RecordDate_SA, 'EEEE') AS Day_Of_Week,

     -- Weekday & Weekend 
     CASE
        WHEN date_format(RecordDate_SA, 'EEEE') IN ('Sunday', 'Saturday') THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_category,

    -- Hour Extraction
    date_format(RecordDate_SA, 'HH:mm:ss') AS Time,


    -- Time of Day classifications
    CASE
        WHEN date_format(RecordDate_SA, 'HH:mm:ss') BETWEEN '00:00:00' AND '11:59:00' THEN 'Morning'
        WHEN date_format(RecordDate_SA, 'HH:mm:ss') BETWEEN '12:00:00' AND '16:59:00' THEN 'Afternoon'
        WHEN date_format(RecordDate_SA, 'HH:mm:ss') BETWEEN '17:00:00' AND '21:59:00' THEN 'Evening'
        ELSE 'Night'
    END AS Time_Group,
    
    -- Viewing hour (individual hour)
    EXTRACT(HOUR FROM RecordDate_SA) AS Viewing_Hour,
    
    -- Sum of viewing hours (aggregated)
    SUM(EXTRACT(HOUR FROM RecordDate_SA)) AS Total_Viewing_Hours,

    --  Month Feature
     
     date_format(RecordDate_SA, 'MMMM') AS Month_name,
    
      -- Time patterns
    CASE
        WHEN day(RecordDate_SA) BETWEEN 1 AND 10 THEN 'Beginning of Month'
        WHEN day(RecordDate_SA) BETWEEN 11 AND 20 THEN 'Mid of Month'
        ELSE 'End of Month'
    END AS month_pattern,
    
   

--===========================================================================================
    -- 3. DURATION METRICS 
--===========================================================================================
    date_format(`Duration 2`, 'HH:mm:ss') AS Duration,
    (EXTRACT(HOUR FROM `Duration 2`) * 60) + EXTRACT(MINUTE FROM `Duration 2`) AS watch_minutes,
    
   -- Movie length
    CASE
        WHEN `Duration 2` BETWEEN '00:00:00' AND '00:04:00' THEN 'Ultra_Short_00:00-00:04'
        WHEN `Duration 2` BETWEEN '00:05:00' AND '00:14:00' THEN 'Quick_Bite_00:05-00:14'
        WHEN `Duration 2` BETWEEN '00:15:00' AND '00:29:00' THEN 'Short_Clip_00:15-00:29'
        WHEN `Duration 2` BETWEEN '00:30:00' AND '00:59:00' THEN 'Standard_View_00:30-00:59'
        WHEN `Duration 2` BETWEEN '01:00:00' AND '01:59:00' THEN 'Extended_View_01:00-01:59'
        WHEN `Duration 2` BETWEEN '02:00:00' AND '02:59:00' THEN 'Deep_Engagement_02:00-02:59'
        WHEN `Duration 2` BETWEEN '03:00:00' AND '03:59:00' THEN 'Marathon_Start_03:00-03:59'
        WHEN `Duration 2` BETWEEN '04:00:00' AND '05:59:00' THEN 'Movie_Length_04:00-05:59'
        ELSE 'Binge_Length'
    END AS Duration_Group,
    
--==============================================================================================
-- 4. CHANNEL METRICS
--==============================================================================================
    channel,
    COUNT(channel) AS Total_Channel_Views,
    COUNT(DISTINCT channel) AS Total_Channels,
    
--==============================================================================================
-- 5. DEMOGRAPHICS
--==============================================================================================
    -- Age
    age,
    CASE
        WHEN age BETWEEN 0 AND 12 THEN 'Children_0-12'
        WHEN age BETWEEN 13 AND 19 THEN 'Teen_13-19'
        WHEN age BETWEEN 20 AND 39 THEN 'Young_Adults_20-39'
        WHEN age BETWEEN 40 AND 59 THEN 'Middle_Aged_Adults_40-59'
        WHEN age >= 60 THEN 'Seniors_60+'
        ELSE 'Unknown'
    END AS Age_Basket,
    ROUND(AVG(age), 0) AS Average_Age,
    
    -- Gender
    gender,
    
    -- Race
    race,
    
    -- Location
    province,
    
--=============================================================================================
    -- 6. VIEWERSHIP METRICS
--=============================================================================================
    COUNT(UserID0) AS Total_Viewership,
    COUNT(DISTINCT UserID0) AS Total_Customers,
    
--=============================================================================================
    -- 7. WATCH TIME AGGREGATIONS (Minute sums removed)
--=============================================================================================
    -- Total viewing hours aggregated
    SUM(EXTRACT(HOUR FROM `Duration 2`)) AS Total_Watch_Hours,
    ROUND(AVG(EXTRACT(HOUR FROM `Duration 2`)), 2) AS Avg_Watch_Hours,
    
    -- Alternative: Total watch time in hours with decimal
    ROUND(SUM((EXTRACT(HOUR FROM `Duration 2`) * 60 + EXTRACT(MINUTE FROM `Duration 2`)) / 60.0), 2) AS Total_Watch_Hours_Decimal,
    ROUND(AVG((EXTRACT(HOUR FROM `Duration 2`) * 60 + EXTRACT(MINUTE FROM `Duration 2`)) / 60.0), 2) AS Avg_Watch_Hours_Decimal,
    
--==============================================================================================
    -- 8. VIEWER CLASSIFICATION (Updated based on hours)
--===============================================================================================
    CASE
        WHEN SUM(EXTRACT(HOUR FROM `Duration 2`)) <= 4 THEN 'Light Viewer (0-4 hours)'
        WHEN SUM(EXTRACT(HOUR FROM `Duration 2`)) <= 8 THEN 'Moderate Viewer (5-8 hours)'
        WHEN SUM(EXTRACT(HOUR FROM `Duration 2`)) <= 12 THEN 'Active Viewer (9-12 hours)'
        WHEN SUM(EXTRACT(HOUR FROM `Duration 2`)) <= 16 THEN 'Heavy Viewer (13-16 hours)'
        ELSE 'Super Viewer (17+ hours)'
    END AS Watch_Category

FROM profile

GROUP BY 
    -- Identifiers
    UserID0,
    
    -- Date & Time Dimensions
    Full_date, 
    Month_name,  
    day_of_week, 
    Time, 
    viewing_hour,
    month_pattern,
    day_category,
    Time_Group,
    time_group,
    
    -- Duration Metrics
    Duration, 
    watch_minutes,
    Duration_Group,
    
    -- Channel
    channel,
    
    -- Demographics
    age,
    Age_Basket,
    gender,
    race,
    province,
    
    -- Raw fields
    RecordDate_SA, 
    `Duration 2`

ORDER BY Full_date, Time;




